# Create the test datastracture for 3 edges

In [1]:
import pandas as pd

# Specify the features to keep
selected_features = [
    'Src IP', 'Dst IP', 'Timestamp', 'Label', 
    'Src Port', 'Dst Port', 'Fwd Header Len', 'Init Bwd Win Byts', 
    'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Init Fwd Win Byts', 
    'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Bwd Pkt Len Mean', 
    'Idle Min', 'Bwd Header Len', 'Pkt Len Var', 'Subflow Fwd Byts', 
    'TotLen Bwd Pkts', 'Idle Max', 'Fwd Seg Size Min', 'Idle Mean', 
    'Pkt Len Max', 'Bwd Pkt Len Std', 'Bwd Pkt Len Max', 
    'Protocol', 'Pkt Len Mean', 'Down/Up Ratio'
]

# Load your dataset
data = pd.read_csv('test_data.csv')  # Replace 'test_data.csv' with your actual file name

# Keep only the specified features
filtered_data = data[selected_features]

# Convert 'Timestamp' to datetime
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'])

# Order the data by 'Timestamp'
filtered_data = filtered_data.sort_values(by='Timestamp')

# Save the temporally ordered data to a new file
filtered_data.to_csv('filtered_test_3edge.csv', index=False)

print("Filtered and temporally ordered data saved to 'filtered_test_3edge.csv'.")


Filtered and temporally ordered data saved to 'filtered_test_3edge.csv'.


In [2]:
#check for inside of csv (just for test, no need for run)
import pandas as pd

# Load the CSV file
file_path = "filtered_test_3edge.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)

# Check if the label column contains '1'
label_column = 'Label'  # Replace with the actual label column name if different
if label_column in df.columns:
    label_distribution = df[label_column].value_counts()
    print("Label Distribution:")
    print(label_distribution)

    if 1 in label_distribution.index:
        print("The CSV contains label '1'.")
    else:
        print("The CSV does NOT contain label '1'.")
else:
    print(f"'{label_column}' column not found in the CSV.")

Label Distribution:
Label
0    730571
1    330799
Name: count, dtype: int64
The CSV contains label '1'.


# created hourly graph with 3 edges from test dataset

In [3]:
import pandas as pd
import networkx as nx
import os
import pickle

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]


def _add_or_update_edge(G, src, dst, edge_key, label, attrs):
    """Add edge or update it; escalate label to 1 if any flow between this pair is an attack."""
    if G.has_edge(src, dst, key=edge_key):
        G[src][dst][edge_key]['label'] = max(G[src][dst][edge_key].get('label', 0), label)
        G[src][dst][edge_key].update(attrs)
    else:
        G.add_edge(src, dst, key=edge_key, label=label, **attrs)


def create_test_graphs_edge_labels(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']

            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            _add_or_update_edge(G, src_ip, dst_ip, 'network', label,
                                {'interaction': 'network_communication',
                                 **{f: row.get(f, 0) for f in NETWORK_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'context', label,
                                {'interaction': 'context',
                                 **{f: row.get(f, 0) for f in CONTEXT_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'knowledge', label,
                                {'interaction': 'knowledge',
                                 **{f: row.get(f, 0) for f in KNOWLEDGE_FEATURES}})

        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

if __name__ == "__main__":
    df_test = pd.read_csv('filtered_test_3edge.csv')
    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    df_test = df_test.set_index('Timestamp').sort_index()
    create_test_graphs_edge_labels(df_test, "3ed_tes_h_graphs")


Hour 0:
Label
1    86671
0    55354
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_tes_h_graphs/test_graph_hour_0.gpickle
Hour 1:
Label
1    214020
0    168193
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_tes_h_graphs/test_graph_hour_1.gpickle
Hour 18:
Label
0    61793
1     1949
Name: count, dtype: int64
Test graph for hour 18 saved to 3ed_tes_h_graphs/test_graph_hour_18.gpickle
Hour 19:
Label
0    760
Name: count, dtype: int64
Test graph for hour 19 saved to 3ed_tes_h_graphs/test_graph_hour_19.gpickle
Hour 20:
Label
0    10276
1      415
Name: count, dtype: int64
Test graph for hour 20 saved to 3ed_tes_h_graphs/test_graph_hour_20.gpickle
Hour 21:
Label
0    5942
1     777
Name: count, dtype: int64
Test graph for hour 21 saved to 3ed_tes_h_graphs/test_graph_hour_21.gpickle
Hour 22:
Label
0    5513
1     785
Name: count, dtype: int64
Test graph for hour 22 saved to 3ed_tes_h_graphs/test_graph_hour_22.gpickle
Hour 23:
Label
0    5013
1     781
Name: count, 

In [ ]:
import pandas as pd
import networkx as nx
import os
import pickle

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]


def _add_or_update_edge(G, src, dst, edge_key, label, attrs):
    """Add edge or update it; escalate label to 1 if any flow between this pair is an attack."""
    if G.has_edge(src, dst, key=edge_key):
        G[src][dst][edge_key]['label'] = max(G[src][dst][edge_key].get('label', 0), label)
        G[src][dst][edge_key].update(attrs)
    else:
        G.add_edge(src, dst, key=edge_key, label=label, **attrs)


def add_node_features(G):
    for node in G.nodes:
        G.nodes[node]['degree'] = G.degree[node]

    undirected_graph = nx.Graph(G)
    communities = nx.community.label_propagation_communities(undirected_graph)
    community_mapping = {
        node: cid
        for cid, community in enumerate(communities)
        for node in community
    }
    for node in G.nodes:
        cid = community_mapping.get(node, -1)
        deg = G.nodes[node]['degree']
        G.nodes[node]['community'] = cid
        G.nodes[node]['x'] = [cid, deg]

    return G


def create_test_graphs_with_node_features(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']

            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            _add_or_update_edge(G, src_ip, dst_ip, 'network', label,
                                {'interaction': 'network_communication',
                                 **{f: row.get(f, 0) for f in NETWORK_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'context', label,
                                {'interaction': 'context',
                                 **{f: row.get(f, 0) for f in CONTEXT_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'knowledge', label,
                                {'interaction': 'knowledge',
                                 **{f: row.get(f, 0) for f in KNOWLEDGE_FEATURES}})

        G = add_node_features(G)

        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

if __name__ == "__main__":
    df_test = pd.read_csv('filtered_test_3edge.csv')
    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    df_test = df_test.set_index('Timestamp').sort_index()
    create_test_graphs_with_node_features(df_test, "3ed_tes_h_graphs")


# Community detection for graphs and then update the graph with the label of community for each node

In [4]:
import networkx as nx
import os
import pickle

def detect_and_label_communities_lpa(graph):
    """
    Run LPA on the undirected projection and write community + degree into each node.
    Node feature vector x = [community_id, degree] (2-D, matches paper section 5.2).
    """
    undirected_graph = nx.Graph(graph)
    communities = nx.community.label_propagation_communities(undirected_graph)

    for community_id, community in enumerate(communities):
        for node in community:
            degree = graph.degree(node)
            graph.nodes[node]['community'] = community_id
            graph.nodes[node]['degree'] = degree
            graph.nodes[node]['x'] = [community_id, degree]

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, "rb") as f:
            G = pickle.load(f)

        G = detect_and_label_communities_lpa(G)

        updated_graph_path = os.path.join(output_dir, graph_file)
        with open(updated_graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Updated graph saved to {updated_graph_path}")


if __name__ == "__main__":
    process_graphs_with_lpa("3ed_tes_h_graphs", "3ed_tes_h_graphs_commun")


Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_21.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_40.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_42.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_34.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_35.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_30.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_49.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_50.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_22.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_36.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_51.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_41.gpickle
Updated graph saved to 3ed_tes_h_graphs_commun/test_graph_hour_18.gpickle
Updated graph saved to 3ed_tes_h_graph

# convert Multigraph to hetrodata

In [5]:
import torch
import os
import pickle
from torch_geometric.data import HeteroData
import networkx as nx

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]
EDGE_FEATURES = {
    'network':   NETWORK_FEATURES,
    'context':   CONTEXT_FEATURES,
    'knowledge': KNOWLEDGE_FEATURES,
}

def multiDiGraph_to_hetero_with_label(G: nx.MultiDiGraph) -> HeteroData:
    """
    Converts a MultiDiGraph to a HeteroData object.
    Node features: x = [community_id, degree]  (2-D)
    Edge features: per-type numeric feature vectors
    """
    data = HeteroData()
    node_mapping = {node: i for i, node in enumerate(G.nodes())}
    data['ip'].num_nodes = G.number_of_nodes()

    community_labels, x = [], []
    for node in G.nodes():
        cid = G.nodes[node].get('community', -1)
        deg = G.nodes[node].get('degree', G.degree(node))
        community_labels.append(cid)
        x.append([cid, deg])
    data['ip'].community = torch.tensor(community_labels, dtype=torch.long)
    data['ip'].x = torch.tensor(x, dtype=torch.float)

    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        src = node_mapping[u]
        dst = node_mapping[v]
        rel_type = ('ip', key, 'ip')
        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr  = []
            data[rel_type].edge_label = []

        data[rel_type].edge_index.append([src, dst])
        feature_vec = [edge_attrs.get(f, 0) for f in EDGE_FEATURES.get(key, [])]
        data[rel_type].edge_attr.append(feature_vec)

        label = edge_attrs.get('label', -1)
        if label == -1:
            print(f"Warning: missing label for edge {u} -> {v} of type {key}")
        data[rel_type].edge_label.append(label)

    for rel_type in data.edge_types:
        data[rel_type].edge_index = (
            torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        )
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(
                data[rel_type].edge_attr, dtype=torch.float
            )
        if data[rel_type].edge_label:
            data[rel_type].edge_label = torch.tensor(
                data[rel_type].edge_label, dtype=torch.long
            )
    return data


def process_and_save_hetero_graphs_with_label(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue
        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)
        hetero_data = multiDiGraph_to_hetero_with_label(G)
        hetero_path = os.path.join(output_dir, graph_file.replace('.gpickle', '.pt'))
        torch.save(hetero_data, hetero_path)
        print(f"Saved HeteroData to {hetero_path}")


if __name__ == "__main__":
    process_and_save_hetero_graphs_with_label(
        "3ed_tes_h_graphs_commun",
        "3ed_tes_h_graphs_hetero_graphs",
    )


/home/rems/code/IoT-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_21.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_40.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_42.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_34.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_35.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_30.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_49.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_50.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_22.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_36.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_51.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_41.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/test_graph_hour_18.pt
Saved HeteroData to 3ed_tes_h_graphs_hetero_graphs/